In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_log_error
from sklearn.ensemble import GradientBoostingRegressor

In [2]:
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")

In [3]:
df = pd.read_csv("Superstore.csv")

In [4]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2013-152156,09-11-13,12-11-13,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2013-152156,09-11-13,12-11-13,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2013-138688,13-06-2013,17-06-2013,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2012-108966,11-10-12,18-10-2012,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2012-108966,11-10-12,18-10-2012,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [5]:
df.isna().sum()

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

In [6]:
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

In [7]:
df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=True)

In [8]:
df['ship_date'] = pd.to_datetime(df['ship_date'], dayfirst=True)

In [9]:
df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['day'] = df['order_date'].dt.day
df['dayofweek'] = df['order_date'].dt.dayofweek
df['quarter'] = df['order_date'].dt.quarter

In [10]:
df['shipping_days'] = (df['ship_date'] - df['order_date']).dt.days

In [11]:
drop_cols =[
    'row_id', 'order_id', 'customer_id',
    'customer_name', 'product_id',
    'ship_date', 'ship_mode', 
    'postal_code', 'region'
]

df = df.drop(columns=drop_cols, errors='ignore')

In [12]:
df.head()

,order_date,segment,country,city,state,category,sub_category,product_name,sales,quantity,discount,profit,year,month,day,dayofweek,quarter,shipping_days
0,2013-11-09,Consumer,United States,Henderson,Kentucky,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2013,11,9,5,4,3
1,2013-11-09,Consumer,United States,Henderson,Kentucky,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2013,11,9,5,4,3
2,2013-06-13,Corporate,United States,Los Angeles,California,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2013,6,13,3,2,4
3,2012-10-11,Consumer,United States,Fort Lauderdale,Florida,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2012,10,11,3,4,7
4,2012-10-11,Consumer,United States,Fort Lauderdale,Florida,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2012,10,11,3,4,7


In [13]:
monthly_sales = df.groupby('order_date')['sales'].sum()

In [14]:
monthly_sales = (
    df.groupby([
        pd.Grouper(key='order_date', freq='M'),
        'state',
        'sub_category'
    ])
    .agg({
        'sales': 'sum',
        'quantity': 'sum',
        'profit': 'sum',
        'discount': 'mean' 
    })
    .reset_index()
)

In [15]:
order_counts = (
    df.groupby([
        pd.Grouper(key='order_date', freq='M'),
        'state',
        'sub_category'
    ])
    .size()
    .reset_index(name='order_count')
)

In [16]:
monthly_sales = monthly_sales.merge(
    order_counts,
    on=['order_date', 'state', 'sub_category'],
    how='left'
)

In [17]:
monthly_sales['price'] = monthly_sales['sales'] / (monthly_sales['quantity'] + 1e-5)

In [18]:
monthly_sales['profit_margin'] = monthly_sales['profit'] / (monthly_sales['sales'] + 1e-5)

In [19]:
def is_peak_season(date):
    month = date.month
    
    if month in [11, 12]:
        return 1
    
    if month in [6, 7]:
        return 1
    
    return 0

monthly_sales['is_peak_season'] = monthly_sales['order_date'].apply(is_peak_season)

In [20]:
def holiday_weight(date):
    month = date.month
    
    if month == 12: 
        return 3
    elif month == 11:  
        return 2
    elif month == 1:  
        return 1
    else:
        return 0

monthly_sales['holiday_impact'] = monthly_sales['order_date'].apply(holiday_weight)

In [21]:
monthly_sales.head()

,order_date,state,sub_category,sales,quantity,profit,discount,order_count,price,profit_margin,is_peak_season,holiday_impact
0,2011-01-31,Arizona,Binders,32.340,10,-23.7160,0.7,1,3.233997,-0.733333,0,1
1,2011-01-31,Arizona,Bookcases,181.470,5,-320.5970,0.7,1,36.293927,-1.766667,0,1
2,2011-01-31,Arizona,Envelopes,108.720,5,36.6930,0.2,1,21.743957,0.337500,0,1
3,2011-01-31,Arizona,Paper,56.064,4,19.6224,0.2,1,14.015965,0.350000,0,1
4,2011-01-31,Arkansas,Art,29.590,10,8.4485,0.0,2,2.958997,0.285519,0,1


In [22]:
monthly_sales['days_to_christmas'] = (
    pd.to_datetime(monthly_sales['order_date'].dt.year.astype(str) + '-12-25')
    - monthly_sales['order_date']
).dt.days

In [23]:
monthly_sales.head()

,order_date,state,sub_category,sales,quantity,profit,discount,order_count,price,profit_margin,is_peak_season,holiday_impact,days_to_christmas
0,2011-01-31,Arizona,Binders,32.340,10,-23.7160,0.7,1,3.233997,-0.733333,0,1,328
1,2011-01-31,Arizona,Bookcases,181.470,5,-320.5970,0.7,1,36.293927,-1.766667,0,1,328
2,2011-01-31,Arizona,Envelopes,108.720,5,36.6930,0.2,1,21.743957,0.337500,0,1,328
3,2011-01-31,Arizona,Paper,56.064,4,19.6224,0.2,1,14.015965,0.350000,0,1,328
4,2011-01-31,Arkansas,Art,29.590,10,8.4485,0.0,2,2.958997,0.285519,0,1,328


In [24]:
monthly_sales['order_lag_1'] = (
    monthly_sales.groupby(['state', 'sub_category'])['order_count'].shift(1)
)

monthly_sales['order_rolling_3'] = (
    monthly_sales.groupby(['state', 'sub_category'])['order_count']
    .transform(
        lambda x: x.shift(1).rolling(3).mean()
    )
)

In [25]:
monthly_sales = monthly_sales.sort_values(['state', 'sub_category', 'order_date',])

monthly_sales['lag_1'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['sales']
    .shift(1)
)

monthly_sales['lag_2'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['sales']
    .shift(2)
)

monthly_sales['lag_12'] = (
    monthly_sales
    .groupby(['state','sub_category'])['sales']
    .shift(12)
)
monthly_sales['rolling_3'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['sales']
    .transform(lambda x: x.rolling(3).mean())
)

In [26]:
monthly_sales['rolling_3'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['sales']
    .transform(lambda x: x.shift(1).rolling(3).mean())
)

monthly_sales['rolling_6'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['sales']
    .transform(lambda x: x.shift(1).rolling(6).mean())
)

monthly_sales['rolling_12'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['sales']
    .transform(lambda x: x.shift(1).rolling(12).mean())
)

In [27]:
monthly_sales['quantity_lag_1'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['quantity']
    .shift(1)
)

monthly_sales['quantity_lag_2'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['quantity']
    .shift(2)
)

monthly_sales['quantity_rolling_3'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['quantity']
    .transform(
        lambda x: x.shift(1).rolling(3).mean()
    )
)

In [28]:
monthly_sales['profit_lag_1'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['profit']
    .shift(1)
)

In [29]:
monthly_sales['discount_lag_1'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['discount']
    .shift(1)
)

In [30]:
monthly_sales['black_friday_flag'] = 0

monthly_sales.loc[
    monthly_sales['order_date'].dt.month == 11,
    'black_friday_flag'
] = 1

In [31]:
monthly_sales['profit_margin_lag_1'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['profit_margin']
    .shift(1)
)

monthly_sales['profit_margin_lag_2'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['profit_margin']
    .shift(2)
)

In [32]:
monthly_sales['order_count_lag_1'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['order_count']
    .shift(1)
)

In [33]:
monthly_sales['order_count_lag_2'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['order_count']
    .shift(2)
)

In [34]:
monthly_sales['order_count_rolling_3'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['order_count']
    .transform(
        lambda x: x.shift(1).rolling(3).mean()
    )
)

In [35]:
monthly_sales['quarter'] = (
    monthly_sales['order_date'].dt.quarter
)

In [36]:
monthly_sales['year'] = (
    monthly_sales['order_date'].dt.year
)

In [37]:
monthly_sales['time_index'] = np.arange(len(monthly_sales))

In [38]:
monthly_sales['month'] = monthly_sales['order_date'].dt.month

In [39]:
monthly_sales['month_sin'] = np.sin(2 * np.pi * monthly_sales['month']/12)
monthly_sales['month_cos'] = np.cos(2 * np.pi * monthly_sales['month']/12)

In [40]:
monthly_sales['momentum_1'] = (
    monthly_sales['lag_1'] - monthly_sales['lag_2']
)

In [41]:
monthly_sales = monthly_sales.dropna()

In [42]:
monthly_sales.head()

,order_date,state,sub_category,sales,quantity,profit,discount,order_count,price,profit_margin,...,order_count_lag_1,order_count_lag_2,order_count_rolling_3,quarter,year,time_index,month,month_sin,month_cos,momentum_1
4095,2014-02-28,Arizona,Accessories,31.992,1,4.7988,0.2,1,31.991680,0.150000,...,1.0,2.0,1.333333,1,2014,69,2,8.660254e-01,5.000000e-01,-49.936
4412,2014-05-31,Arizona,Accessories,271.968,4,54.3936,0.2,1,67.991830,0.200000,...,1.0,1.0,1.333333,2,2014,70,5,5.000000e-01,-8.660254e-01,-63.992
4571,2014-06-30,Arizona,Accessories,89.544,7,12.3123,0.2,1,12.791982,0.137500,...,1.0,1.0,1.000000,2,2014,71,6,1.224647e-16,-1.000000e+00,239.976
4976,2014-09-30,Arizona,Accessories,89.584,2,4.4792,0.2,1,44.791776,0.050000,...,1.0,1.0,1.000000,3,2014,72,9,-1.000000e+00,-1.836970e-16,-182.424
5347,2014-11-30,Arizona,Accessories,343.904,16,-33.9044,0.2,3,21.493987,-0.098587,...,1.0,1.0,1.000000,4,2014,73,11,-5.000000e-01,8.660254e-01,0.040


In [43]:
monthly_sales.columns

Index(['order_date', 'state', 'sub_category', 'sales', 'quantity', 'profit',
       'discount', 'order_count', 'price', 'profit_margin', 'is_peak_season',
       'holiday_impact', 'days_to_christmas', 'order_lag_1', 'order_rolling_3',
       'lag_1', 'lag_2', 'lag_12', 'rolling_3', 'rolling_6', 'rolling_12',
       'quantity_lag_1', 'quantity_lag_2', 'quantity_rolling_3',
       'profit_lag_1', 'discount_lag_1', 'black_friday_flag',
       'profit_margin_lag_1', 'profit_margin_lag_2', 'order_count_lag_1',
       'order_count_lag_2', 'order_count_rolling_3', 'quarter', 'year',
       'time_index', 'month', 'month_sin', 'month_cos', 'momentum_1'],
      dtype='object')

In [44]:
df['order_date'].min(), df['order_date'].max()

(Timestamp('2011-01-04 00:00:00'), Timestamp('2014-12-31 00:00:00'))

In [45]:
train = monthly_sales[monthly_sales['order_date'] < '2014-12-31']
test = monthly_sales[monthly_sales['order_date'] >= '2014-12-31']

In [46]:
target = 'sales'

categorical_cols = ['sub_category', 'state']
numerical_cols = [ 'quantity', 'profit', 'discount', 'order_count', 'price', 'profit_margin', 'lag_1', 'lag_2', 'lag_12', 'rolling_3', 'rolling_6', 'rolling_12', 'quantity_lag_1', 'quantity_lag_2', 'quantity_rolling_3', 'profit_lag_1', 'discount_lag_1', 'time_index', 'month', 'quarter', 'year', 'month_sin', 'month_cos', 'momentum_1', 'is_peak_season', 'holiday_impact', 'days_to_christmas', 'black_friday_flag' ]

X_train = train[categorical_cols + numerical_cols]
y_train = train[target]

X_test = test[categorical_cols + numerical_cols]
y_test = test[target]

In [47]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [48]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numerical_cols)
    ]
)

In [65]:
!pip install catboost

In [49]:
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=1500,
    depth=5,
    learning_rate=0.05,
    l2_leaf_reg=5,
    random_state=42,
    verbose=200
) 

In [50]:
pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ]
)

In [51]:
pipeline.fit(X_train, y_train)

0:	learn: 812.3605323	total: 153ms	remaining: 3m 49s
200:	learn: 116.5272169	total: 1.02s	remaining: 6.59s
400:	learn: 51.7857003	total: 1.91s	remaining: 5.24s
600:	learn: 31.6143921	total: 2.74s	remaining: 4.09s
800:	learn: 22.2564134	total: 3.49s	remaining: 3.04s
1000:	learn: 17.0767785	total: 4.15s	remaining: 2.07s
1200:	learn: 13.5213394	total: 4.82s	remaining: 1.2s
1400:	learn: 11.0112754	total: 5.53s	remaining: 391ms
1499:	learn: 10.0225366	total: 5.85s	remaining: 0us


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [52]:
from sklearn.metrics import mean_absolute_error, r2_score

preds = pipeline.predict(X_test)

print("MAE:", mean_absolute_error(y_test, preds))
print("R2:", r2_score(y_test, preds))

MAE: 39.93836200149944
R2: 0.9881492588747538


In [53]:
preds = np.maximum(preds, 0)

In [54]:
from sklearn.metrics import mean_squared_log_error
rmsle = np.sqrt(mean_squared_log_error(y_test, preds))
print("RMSLE:", rmsle)

RMSLE: 0.2581813268249875


In [55]:
residuals = y_test - preds

error_std = residuals.std()

print("Forecast Error Std:", error_std)

Forecast Error Std: 79.84310300719528


In [56]:
lower_bound = preds - 1.96 * error_std
upper_bound = preds + 1.96 * error_std

In [58]:
joblib.dump(pipeline, "sales_model.pkl")

['sales_model.pkl']

In [109]:
feature_importance = pipeline.named_steps['model'].get_feature_importance()

feature_names = (
    pipeline.named_steps['preprocessor']
    .get_feature_names_out()
)

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print(importance_df.head(20))

                     feature  importance
40                num__price   39.932022
36             num__quantity   23.202359
37               num__profit   21.025436
41        num__profit_margin    3.985781
39          num__order_count    1.774881
47           num__rolling_12    1.614663
55           num__time_index    0.655333
53  num__profit_margin_lag_1    0.646561
43                num__lag_2    0.531557
59            num__month_sin    0.511503
45            num__rolling_3    0.500001
52       num__discount_lag_1    0.472176
44               num__lag_12    0.464618
60            num__month_cos    0.451780
64    num__days_to_christmas    0.398073
51         num__profit_lag_1    0.351919
49       num__quantity_lag_2    0.350755
46            num__rolling_6    0.348321
42                num__lag_1    0.320628
61           num__momentum_1    0.258453
